## Step 1: Bronze Layer -- CSV to Delta Lake

**Bronze layer purpose**: Read raw CSV from data source, write losslessly to Delta Lake.

**Design principles**:
- Only format conversion (CSV to Delta), no cleaning, dedup, or joins
- Use inferSchema for automatic type inference (preserve raw data shape)
- Store as Delta for ACID transactions + time travel + schema evolution
- Cleaning and standardization left to Silver layer

**Data source**: Databricks Volumes (`/Volumes/workspace/default/olist_files`)

**Output**: 9 Delta tables in `default` schema

In [0]:
# Data source root path -- Databricks Volumes
VOLUME_BASE = "/Volumes/workspace/default/olist_files"

# CSV filename -> Bronze Delta table name mapping
TABLE_MAPPING = {
    "olist_orders_dataset.csv":              "bronze_orders",
    "olist_customers_dataset.csv":           "bronze_customers",
    "olist_order_items_dataset.csv":         "bronze_order_items",
    "olist_products_dataset.csv":            "bronze_products",
    "olist_sellers_dataset.csv":             "bronze_sellers",
    "olist_order_payments_dataset.csv":      "bronze_payments",
    "olist_order_reviews_dataset.csv":       "bronze_reviews",
    "olist_geolocation_dataset.csv":         "bronze_geolocation",
    "product_category_name_translation.csv": "bronze_translation",
}

print(f"Data source: {VOLUME_BASE}")
print(f"Tables to import: {len(TABLE_MAPPING)}")

In [0]:
def ingest_csv_to_bronze(file_name, table_name):
    """
    Read a single CSV from Volumes and write to Delta Lake Bronze table.
    
    Args:
        file_name: CSV filename (e.g. "olist_orders_dataset.csv")
        table_name: Target Delta table name (e.g. "bronze_orders")
    """
    full_path = f"{VOLUME_BASE}/{file_name}"
    print(f"Reading: {full_path}")

    # Read CSV with auto schema inference, no cleaning
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(full_path)

    # Write to Delta Lake (overwrite -- Olist is static snapshot)
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

    row_count = df.count()
    print(f"Done: {table_name} -- {row_count:,} rows")

In [0]:
# Batch ingest all 9 tables
for file_name, table_name in TABLE_MAPPING.items():
    ingest_csv_to_bronze(file_name, table_name)

print("\nAll 9 Bronze tables written successfully!")

In [0]:
# Verify: list all Bronze tables
spark.sql("SHOW TABLES IN default LIKE 'bronze_*'").show(20, False)

---

### Next step

After Bronze tables are created, go to 02_silver for cleaning and joins.